# Import Library

## 
Files required 
1) gamma code = gamma(folder)
2) policy model = policy_model_discreteshift_final_3L_512H_s1_c3.pth
3) reference data = data(folder)
4) moving_average.py
5) GAMMA_obj_temp_depth.py

In [13]:
import numpy as np
import pandas as pd
from typing import Optional, Tuple
import sys
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from pickle import dump
from sklearn.preprocessing import MinMaxScaler
import time
from tqdm import tqdm
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os

# Resolve the primary compute device once and reuse it throughout the notebook.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from moving_average import moving_average_1d
import copy

from GAMMA_obj_temp_depth import GAMMA_obj


Using device: cuda:0


# Import Policy Model

In [14]:
from policy import PolicyNN
import torch

# confirm which devices are available
print(torch.cuda.device_count())  # should be ≥1 to use CUDA

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# or force CPU: device = torch.device('cpu')

model = PolicyNN(
    past_input_dim=6,
    future_input_dim=6,
    output_dim=1,
    p=50,
    window=50,
    hidden_dim=512,
    n_layers=3,
    dropout_p=0.1
).to(device)

state = torch.load(
    "/home/ftk3187/github/DPC_research/02_DED/4_policy_0725/trainresults/policy_model_discreteshift_final_3L_512H_s1_c3.pth",
    map_location=device,
)
model.load_state_dict(state)
model.eval()

1


PolicyNN(
  (input_layer): Linear(in_features=600, out_features=512, bias=True)
  (input_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=512, out_features=512, bias=True)
  )
  (norm_layers): ModuleList(
    (0): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (output_layer): Linear(in_features=512, out_features=50, bias=True)
)

In [15]:
import math

class KFACLaplaceOnline:
    """Kronecker-factored Laplace approximation that updates online for PolicyNN."""

    def __init__(self, model: PolicyNN, prior_precision: float = 1.0, likelihood_std: float = 0.05, damping: float = 1e-3):
        self.model = model
        self.model.eval()
        self.prior_precision = prior_precision
        self.likelihood_std = likelihood_std
        self.damping = damping

        self.layers = [module for module in self.model.modules() if isinstance(module, nn.Linear)]
        if not self.layers:
            raise ValueError('PolicyNN must contain linear layers for KFAC Laplace.')

        self.activations = {layer: None for layer in self.layers}
        self.backprops = {layer: None for layer in self.layers}
        self.layer_stats = {
            layer: {
                'A': torch.zeros((layer.in_features, layer.in_features), dtype=torch.float64, device=layer.weight.device),
                'G': torch.zeros((layer.out_features, layer.out_features), dtype=torch.float64, device=layer.weight.device),
            }
            for layer in self.layers
        }
        self.layer_covariances = {}
        self.sample_count = 0
        self.output_count = 0

        self._register_hooks()

    def _register_hooks(self):
        def make_forward_hook(layer):
            def _hook(module, inputs, output):
                self.activations[layer] = inputs[0].detach()
            return _hook

        def make_backward_hook(layer):
            def _hook(module, grad_input, grad_output):
                self.backprops[layer] = grad_output[0].detach()
            return _hook

        for layer in self.layers:
            layer.register_forward_hook(make_forward_hook(layer))
            layer.register_full_backward_hook(make_backward_hook(layer))

    def _clear_backprops(self):
        for layer in self.layers:
            self.backprops[layer] = None

    def _update_covariances(self):
        if self.sample_count == 0 or self.output_count == 0:
            return
        for layer in self.layers:
            A_mean = self.layer_stats[layer]['A'] / self.sample_count
            G_mean = self.layer_stats[layer]['G'] / self.output_count

            in_dim = layer.in_features
            out_dim = layer.out_features

            eye_in = torch.eye(in_dim, dtype=torch.float64, device=A_mean.device)
            eye_out = torch.eye(out_dim, dtype=torch.float64, device=G_mean.device)

            A_damped = A_mean + (self.prior_precision + self.damping) * eye_in
            G_damped = G_mean + (self.prior_precision + self.damping) * eye_out

            self.layer_covariances[layer] = {
                'A': torch.linalg.inv(A_damped),
                'G': torch.linalg.inv(G_damped)
            }

    def evaluate(self, policy_past: torch.Tensor, policy_future: torch.Tensor, update_stats: bool = True):
        """Return mean control trajectory plus epistemic/total variance estimates."""
        self._clear_backprops()
        self.model.zero_grad(set_to_none=True)

        outputs = self.model((policy_past, policy_future))
        out_flat = outputs.view(outputs.shape[0], -1)
        batch_size, num_outputs = out_flat.shape

        mean = outputs.detach()
        epistemic_var = None
        total_var = None

        if self.layer_covariances:
            var_accum = torch.zeros(batch_size, num_outputs, dtype=torch.float64, device=outputs.device)
            for out_idx in range(num_outputs):
                grad_outputs = torch.zeros_like(out_flat)
                grad_outputs[:, out_idx] = 1.0
                self._clear_backprops()
                self.model.zero_grad(set_to_none=True)
                out_flat.backward(grad_outputs, retain_graph=True)

                for layer in self.layers:
                    a = self.activations[layer].to(torch.float64)
                    delta = self.backprops[layer].to(torch.float64)
                    A_cov = self.layer_covariances[layer]['A']
                    G_cov = self.layer_covariances[layer]['G']
                    delta_term = torch.einsum('bi,ij,bj->b', delta, G_cov, delta)
                    a_term = torch.einsum('bi,ij,bj->b', a, A_cov, a)
                    var_accum[:, out_idx] += delta_term * a_term

            epistemic_var = torch.clamp(var_accum.view_as(outputs).to(outputs.dtype), min=1e-12)
            total_var = epistemic_var + (self.likelihood_std ** 2)

        if update_stats:
            scale = 1.0 / (self.likelihood_std ** 2)
            scale_sqrt = math.sqrt(scale)

            with torch.no_grad():
                for layer in self.layers:
                    a = self.activations[layer].to(torch.float64)
                    self.layer_stats[layer]['A'] += a.transpose(0, 1) @ a

            for out_idx in range(num_outputs):
                grad_outputs = torch.zeros_like(out_flat)
                grad_outputs[:, out_idx] = scale_sqrt
                self._clear_backprops()
                self.model.zero_grad(set_to_none=True)
                retain = out_idx < (num_outputs - 1)
                out_flat.backward(grad_outputs, retain_graph=retain)

                for layer in self.layers:
                    delta = self.backprops[layer].to(torch.float64)
                    self.layer_stats[layer]['G'] += delta.transpose(0, 1) @ delta

            self.sample_count += batch_size
            self.output_count += batch_size * num_outputs
            self._update_covariances()

        self.model.zero_grad(set_to_none=True)
        self._clear_backprops()
        return mean, epistemic_var, total_var



In [16]:
kfac_laplace = KFACLaplaceOnline(model, prior_precision=1.0, likelihood_std=0.05, damping=1e-3)

uncertainty_log = {
    'mean_control': [],
    'epistemic_var_scaled': [],
    'epistemic_var_original': [],
    'total_var_scaled': [],
    'total_var_original': []
}


# Import Reference Data

In [17]:
import cupy as cp

device_count = cp.cuda.runtime.getDeviceCount()
if device_count == 0:
    raise RuntimeError('No CUDA devices detected for GAMMA simulation.')

cp.cuda.Device(0).use()


<CUDA Device 0>

In [18]:
import cupy as cp
device_id = min(0, cp.cuda.runtime.getDeviceCount() - 1)  # pick GPU 0 by default
cp.cuda.Device(device_id).use()

INPUT_DATA_DIR = "data"
SIM_DIR_NAME = "single_track_square"
BASE_LASER_FILE_DIR = "laser_power_profiles/csv"
CLOUD_TARGET_BASE_PATH = "result"
solidus_temp = 1600
window = 50
sim_interval = 5
init_runs = 50 #50 

GAMMA_class = GAMMA_obj(INPUT_DATA_DIR, SIM_DIR_NAME, BASE_LASER_FILE_DIR, CLOUD_TARGET_BASE_PATH, solidus_temp, window, init_runs, sim_interval, laser_power_number=1)
init_avg = GAMMA_class.run_initial_steps()
init_avg = torch.tensor(init_avg,dtype=torch.float32)[:,-window:] # shape = [2,50]

100%|██████████| 250/250 [00:05<00:00, 46.11it/s]


In [19]:
df_one_print = pd.read_csv('single_track_ref.csv')

loc_X_list = df_one_print["X"].to_numpy().reshape(-1,1)
loc_Y_list = df_one_print["Y"].to_numpy().reshape(-1,1)
loc_Z_list = df_one_print["Z"].to_numpy().reshape(-1,1)
dist_X_list = df_one_print["Dist_to_nearest_X"].to_numpy().reshape(-1,1)
dist_Y_list = df_one_print["Dist_to_nearest_Y"].to_numpy().reshape(-1,1)
scan_spd_list = df_one_print["scanning_speed"].to_numpy().reshape(-1,1)

# laser power
laser_power_ref = torch.tensor(df_one_print["Laser_power"].to_numpy().reshape(-1,1),dtype=torch.float32)
laser_power_past = laser_power_ref[:window]
fix_covariates = torch.tensor(np.concatenate((loc_Z_list,dist_X_list,dist_Y_list),axis=1),dtype=torch.float32)

# apply moving average for mp temp
mp_temp_raw = df_one_print["melt_pool_temperature"].to_numpy()
mp_temp_mv = moving_average_1d(mp_temp_raw,4)
mp_temp = copy.deepcopy(mp_temp_raw)
mp_temp[1:-2] = mp_temp_mv
mp_temp = mp_temp

mp_temp_ref = torch.tensor(mp_temp,dtype=torch.float32)

x_min = torch.tensor([[0.0, 0.75, 0.75, 504.26]], dtype=torch.float32).to(device)
x_max = torch.tensor([[7.5, 20.0, 20.0, 732.298]], dtype=torch.float32).to(device)

y_min = torch.tensor([[436.608, -0.559]], dtype=torch.float32).to(device)
y_max = torch.tensor([[4509.855, 0.551]], dtype=torch.float32).to(device)

# Precompute constants that map control values between scaled and original units.
LASER_SCALE = float(0.5 * (x_max[0, 3].item() - x_min[0, 3].item()))
LASER_OFFSET = float(x_min[0, 3].item())


In [20]:
x_min = torch.tensor([[0.0, 0.75, 0.75, 504.26]], dtype=torch.float32).to(device)
x_max = torch.tensor([[7.5, 20.0, 20.0, 732.298]], dtype=torch.float32).to(device)

y_min = torch.tensor([[436.608, -0.559]], dtype=torch.float32).to(device)
y_max = torch.tensor([[4509.855, 0.551]], dtype=torch.float32).to(device)


In [21]:
def normalize_x(x, dim_id):
    x_min_selected = x_min[0, dim_id]
    x_max_selected = x_max[0, dim_id]
    return 2 * (x - x_min_selected) / (x_max_selected - x_min_selected) - 1

def inverse_normalize_x(x_norm, dim_id):
    x_min_selected = x_min[0, dim_id]
    x_max_selected = x_max[0, dim_id]
    return 0.5 * (x_norm + 1) * (x_max_selected - x_min_selected) + x_min_selected

def normalize_y(y, dim_id):
    y_min_selected = y_min[0, dim_id]
    y_max_selected = y_max[0, dim_id]
    return 2 * (y - y_min_selected) / (y_max_selected - y_min_selected) - 1

def inverse_normalize_y(y_norm, dim_id):
    y_min_selected = y_min[0, dim_id]
    y_max_selected = y_max[0, dim_id]
    return 0.5 * (y_norm + 1) * (y_max_selected - y_min_selected) + y_min_selected


# Sub-Function ; Run Policy

In [22]:
import os
import copy
from pathlib import Path

def clone_gamma(G):
    return copy.deepcopy(G)

def rollout_future(G_clone, control_seq):
    temps, depths = [], []
    for u in control_seq:
        x, d = G_clone.run_sim_interval(float(u))
        temps.append(x)
        depths.append(d)
    return temps, depths

def run_one_step_policy(GAMMA_obj, policy_model, P, window, laplace=None, uncertainty_log=None):
    # ===== 1) 입력 구성 =====
    mp_temp_ref = GAMMA_obj.ref[GAMMA_obj.MPC_counter : GAMMA_obj.MPC_counter + P]
    mp_temp_ref_t = torch.as_tensor(mp_temp_ref, dtype=torch.float32, device=device).reshape(1, P, 1)

    mp_temp_past_t = GAMMA_obj.x_past.T.unsqueeze(0).to(device)                    # (1,window,2)
    laser_past_t   = GAMMA_obj.u_past.view(1, -1, 1).to(device)                    # (1,window,1)

    fix_cov_past   = GAMMA_obj.fix_cov_all[GAMMA_obj.MPC_counter - window : GAMMA_obj.MPC_counter, :]
    fix_cov_past_t = torch.as_tensor(fix_cov_past, dtype=torch.float32, device=device).unsqueeze(0)

    fix_cov_past_s = normalize_x(fix_cov_past_t, dim_id=[0,1,2])
    laser_past_s   = normalize_x(laser_past_t,   dim_id=[3])
    mp_temp_past_s = normalize_y(mp_temp_past_t, dim_id=[0,1])
    policy_in_past = torch.cat((fix_cov_past_s, laser_past_s, mp_temp_past_s), dim=2)  # (1,window,?=6)

    fix_cov_future   = GAMMA_obj.fix_cov_all[GAMMA_obj.MPC_counter : GAMMA_obj.MPC_counter + P, :]
    fix_cov_future_t = torch.as_tensor(fix_cov_future, dtype=torch.float32, device=device).unsqueeze(0)
    fix_cov_future_s = normalize_x(fix_cov_future_t, dim_id=[0,1,2])

    mp_temp_ref_s = normalize_y(mp_temp_ref_t, dim_id=[0])[:, :, 0].unsqueeze(-1)

    depth_lower_const = 0.1423
    depth_upper_const = 0.4126
    y_const_s = torch.tensor([[depth_lower_const, depth_upper_const]] * P, dtype=torch.float32, device=device).reshape(1, P, 2)

    policy_in_future = torch.cat((fix_cov_future_s, mp_temp_ref_s, y_const_s), dim=2)  # (1,P,?=6)

    # ===== 2) 정책/라플라스 예측 =====
    if laplace is not None:
        u_pred, epistemic_var, total_var = laplace.evaluate(policy_in_past, policy_in_future, update_stats=True)  # shapes (1,P,1)
    else:
        u_pred = policy_model((policy_in_past, policy_in_future))
        epistemic_var = None
        total_var     = None

    # ===== 3) (옵션) 불확실성 past-로그 기록: '이번 스텝의 첫 액션'만 과거 로그로 축적 =====
    if uncertainty_log is not None:
        laser_span   = (x_max[0, 3] - x_min[0, 3]).item()
        laser_scale  = 0.5 * laser_span
        laser_offset = x_min[0, 3].item()

        # 첫 액션의 mean만 past 로그에 (히스토리)
        control_scaled   = u_pred[0, 0, 0].detach().cpu()
        control_original = float((control_scaled + 1.0) * laser_scale + laser_offset)
        uncertainty_log['mean_control'].append(control_original)

        if (epistemic_var is not None) and (total_var is not None):
            var0 = float(epistemic_var[0, 0, 0].detach().cpu())
            tot0 = float(total_var[0, 0, 0].detach().cpu())
            uncertainty_log['epistemic_var_scaled'].append(var0)
            uncertainty_log['total_var_scaled'].append(tot0)
            uncertainty_log['epistemic_var_original'].append((laser_scale**2) * var0)
            uncertainty_log['total_var_original'].append((laser_scale**2) * tot0)
        else:
            uncertainty_log['epistemic_var_scaled'].append(None)
            uncertainty_log['total_var_scaled'].append(None)
            uncertainty_log['epistemic_var_original'].append(None)
            uncertainty_log['total_var_original'].append(None)

    # ===== 4) 🔴 여기서 '미래 horizon 밴드'는 로그가 아니라 현재 evaluate 결과로 만든다 =====
    #      (제어 적용/카운터 증가 이전에 미리 계산·저장·플롯)
    k_before = GAMMA_obj.MPC_counter  # 기준 시점

    # (a) 미래 control mean (물리 단위)
    mean_scaled_h = u_pred[0, :, 0].detach().cpu().numpy()                 # (P,)
    laser_span   = (x_max[0, 3] - x_min[0, 3]).item()
    laser_scale  = 0.5 * laser_span
    laser_offset = x_min[0, 3].item()
    mean_h = (mean_scaled_h + 1.0) * laser_scale + laser_offset            # (P,)

    # (b) 미래 control 분산 (물리 단위)  ← 로그(X) 말고 evaluate 결과 사용!!
    if (epistemic_var is not None) and (total_var is not None):
        tot_h = total_var[0, :, 0].detach().cpu().numpy()                  # (P,)
        epi_h = epistemic_var[0, :, 0].detach().cpu().numpy()              # (P,)
        alea_h_scaled = np.maximum(tot_h - epi_h, 0.0)                     # (P,) (scaled space)
        var_h = (laser_scale**2) * alea_h_scaled                           # (P,) (physical)
        std_h = np.sqrt(var_h)
        upper_h = mean_h + 1.28 * std_h
        lower_h = mean_h - 1.28 * std_h
    else:
        upper_h = mean_h.copy()
        lower_h = mean_h.copy()

    # (c) 미래 가상 롤아웃 (온도/깊이 upper/lower)
    G_up  = clone_gamma(GAMMA_obj)
    G_low = clone_gamma(GAMMA_obj)
    fut_temp_up,  fut_depth_up  = rollout_future(G_up,  upper_h)
    fut_temp_low, fut_depth_low = rollout_future(G_low, lower_h)

    GAMMA_obj.future_temp_upper = fut_temp_up
    GAMMA_obj.future_temp_lower = fut_temp_low
    GAMMA_obj.future_depth_upper = fut_depth_up
    GAMMA_obj.future_depth_lower = fut_depth_low

    # (d) 저장/플롯 (원하면 주기적으로)
    if (k_before + 1) % 50 == 0:
        save_dir = Path("plots"); save_dir.mkdir(parents=True, exist_ok=True)
        fig_path = save_dir / f"mpc_step_{k_before:04d}.png"
        plot_fig(MPC_GAMMA=GAMMA_obj, i=k_before)  # 기준 시점 = k_before
        plt.savefig(fig_path, dpi=200); plt.close()
        print(f"[INFO] Saved rollout plot at k={k_before}: {fig_path.resolve()}")

    # ===== 5) 이제 '제어 적용' 및 상태 업데이트 =====
    u_first   = u_pred[0, 0]
    u_applied = float(inverse_normalize_x(u_first, dim_id=[3]))
    x_current, depth_current = GAMMA_obj.run_sim_interval(u_applied)

    GAMMA_obj.x_past[:, :-1] = GAMMA_obj.x_past[:, 1:]
    GAMMA_obj.x_past[0, -1]  = x_current
    GAMMA_obj.x_past[1, -1]  = depth_current
    GAMMA_obj.u_past[:-1]    = GAMMA_obj.u_past[1:].clone()
    GAMMA_obj.u_past[-1]     = u_applied

    GAMMA_obj.x_hat_current  = torch.tensor([x_current, depth_current], device=device)
    GAMMA_obj.x_sys_current  = torch.tensor([[x_current], [depth_current]], device=device)

    GAMMA_obj.MPC_counter += 1

    new_state = torch.tensor([[x_current, depth_current]], device=GAMMA_obj.x_past_save.device)
    GAMMA_obj.x_past_save = torch.cat((GAMMA_obj.x_past_save, new_state), dim=0)
    new_u = torch.tensor([[u_applied]], device=GAMMA_obj.u_past_save.device)
    GAMMA_obj.u_past_save = torch.cat((GAMMA_obj.u_past_save, new_u), dim=0)


# Sub-Function ; Plot Rollout

In [23]:
def plot_fig(MPC_GAMMA, i, horizon=50):
    import numpy as np
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=[12,10])

    # ===== Past & Future time axes =====
    past_len = len(MPC_GAMMA.x_past_save)
    t_past = np.arange(past_len)
    t_future = np.arange(i, i + horizon)

    # ===================== Temperature =====================
    plt.subplot(3,1,1)

    # Future predictions
    if hasattr(MPC_GAMMA, "future_temp_upper"):
        fut_h = min(horizon, len(MPC_GAMMA.future_temp_upper))
        plt.plot(t_future[:fut_h], MPC_GAMMA.future_temp_upper[:fut_h], 'r--', label="Future upper")
        plt.plot(t_future[:fut_h], MPC_GAMMA.future_temp_lower[:fut_h], 'g--', label="Future lower")

    # Past data
    plt.plot(t_past, MPC_GAMMA.x_past_save[:,0], label="GAMMA simulation", color="blue")
    plt.plot(t_past, MPC_GAMMA.ref[:past_len], label="Reference", color="orange")

    # Future ref
    if hasattr(MPC_GAMMA, "ref"):
        fut_h_ref = min(horizon, len(MPC_GAMMA.ref) - i)
        if fut_h_ref > 0:
            plt.plot(t_future[:fut_h_ref], MPC_GAMMA.ref[i:i+fut_h_ref],
                     color="orange", alpha=0.8, label="Future ref")

    plt.ylabel("MP Temp (K)")
    plt.title("MPC Temperature & Prediction")
    plt.legend()
    plt.xlim(i-100, i+60)

    # ===================== Depth =====================
    plt.subplot(3,1,2)

    if hasattr(MPC_GAMMA, "future_depth_upper"):
        fut_h = min(horizon, len(MPC_GAMMA.future_depth_upper))
        plt.plot(t_future[:fut_h], MPC_GAMMA.future_depth_upper[:fut_h], 'r--', label="Future upper")
        plt.plot(t_future[:fut_h], MPC_GAMMA.future_depth_lower[:fut_h], 'g--', label="Future lower")

    plt.plot(t_past, MPC_GAMMA.x_past_save[:,1], label="GAMMA simulation", color="blue")

    UB, LB = 0.225, 0.075
    plt.plot(t_past, UB*np.ones(past_len), label="UB (past)", color="orange")
    plt.plot(t_past, LB*np.ones(past_len), label="LB (past)", color="green")

    plt.plot(t_future, UB*np.ones(horizon), color="orange", alpha=0.8, label="UB (future)")
    plt.plot(t_future, LB*np.ones(horizon), color="green", alpha=0.8, label="LB (future)")

    plt.ylabel("MP Depth (mm)")
    plt.title("MPC Depth & Constraints")
    plt.legend()
    plt.xlim(i-100, i+60)

    # ===================== Laser Power =====================
    plt.subplot(3,1,3)

    # ------ Future uncertainty band ------
    if hasattr(MPC_GAMMA, "future_control_mean"):
        fut_h = min(horizon, len(MPC_GAMMA.future_control_mean))
        t_f = t_future[:fut_h]
        plt.fill_between(t_f, MPC_GAMMA.future_control_lower[:fut_h],
                               MPC_GAMMA.future_control_upper[:fut_h],
                               alpha=0.3, color='orange', label="90% CI (future)")
        plt.plot(t_f, MPC_GAMMA.future_control_mean[:fut_h], 'k--', label="Future control mean")

    # ------ Past applied control ------
    plt.plot(t_past, MPC_GAMMA.u_past_save[:past_len], label="Laser power (applied)", color="blue")

    plt.ylabel("Laser power (W)")
    plt.xlabel("MPC time step (0.0355 sec/iteration)")
    plt.title("Laser Power")
    plt.legend()
    plt.xlim(i-100, i+60)

    plt.tight_layout()
    return plt


# Execution

In [ ]:
# step #
P = 50
N_step = len(mp_temp_ref) - init_runs + 50


# initialize GAMMA class
GAMMA_class.ref = mp_temp_ref
GAMMA_class.fix_cov_all = fix_covariates
GAMMA_class.x_past = init_avg.clone()
GAMMA_class.u_past = laser_power_past.clone()

GAMMA_class.x_hat_current = GAMMA_class.x_past[:, -1]
GAMMA_class.x_sys_current = GAMMA_class.x_past[:, -1].reshape(2, 1)

GAMMA_class.x_past_save = GAMMA_class.x_past.T.clone()
GAMMA_class.u_past_save = GAMMA_class.u_past.clone()
GAMMA_class.MPC_counter = window 


# execution loop
from tqdm import tqdm

for i in tqdm(range(N_step)):
    run_one_step_policy(GAMMA_class, model, P=P, window=window, laplace=kfac_laplace, uncertainty_log=uncertainty_log)

    if i % 10 == 0:
        plot_fig(GAMMA_class, i)


  1%|          | 38/6295 [07:21<20:25:02, 11.75s/it]

# Inspect KFAC Uncertainty


In [ ]:
uncertainty_df = pd.DataFrame(uncertainty_log)
uncertainty_df.head()
